#### Import Libraries

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("All good 🚀")

All good 🚀


##### Load the dataset

In [74]:
model_df = pd.read_csv("../data/processed/bed_occupancy_modelling_dataset.csv")

#### Create new features

In [75]:
model_df["over_capacity"] = (
    model_df["occupancy_rate"] > 1
).astype(int)

##### cap occupancy rate

In [76]:
model_df["occupancy_rate_model"] = (
    model_df["occupancy_rate"]
    .clip(0,1)
)

##### bed pressure feature

In [77]:
model_df["bed_pressure"] = (
    model_df["occupied_beds"] -
    model_df["staffed_beds"]
)

In [78]:
model_df["available_beds"] = (
    model_df["available_beds"]
    .clip(lower=0)
)

#### Sort Date Column

In [79]:
model_df["date"] = pd.to_datetime(
    model_df["date"]
)

In [80]:
model_df = model_df.sort_values(
    [
        "hospital_id",
        "ward",
        "date"
    ]
)

In [81]:
model_df.duplicated(
    [
        "hospital_id",
        "ward",
        "date"
    ]
).sum()

0

##### Check Date Coverage - continuous dates for forecasting

In [82]:
model_df.groupby(
    [
        "hospital_id",
        "ward"
    ]
)["date"].agg(
    ["min","max","count"]
)

min        max  count
hospital_id ward                                                
HHN-BIR-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-EDI-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-LON-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-LON-02  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-MAN-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731

##### Add Time-Series Features

##### Lag 1 - Yesterday's occupancy

In [83]:
model_df["lag_1_occupancy"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupancy_rate_model"]
    .shift(1)
)

##### Lag 7 - One week ago (Previous 7 days)

In [84]:
model_df["lag_7_occupancy"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupancy_rate_model"]
    .shift(7)
)

##### 7-day moving average

In [85]:
model_df["rolling_7_day_avg"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupancy_rate_model"]
    .transform(
        lambda x:
        x.rolling(7).mean()
    )
)

##### Day of week

In [86]:
model_df["day_of_week"] = (
    model_df["date"]
    .dt.dayofweek
)

##### Day-to-Day Change

In [87]:
model_df["occupancy_change"] = (
    model_df.groupby(
        [
            "hospital_id",
            "ward"
        ]
    )["occupancy_rate"]
    .diff()
)

##### Fill Lag NA Values

In [88]:
lag_columns = [
    "lag_1_occupancy",
    "lag_7_occupancy",
    "rolling_7_day_avg",
    "occupancy_change"
]

model_df[lag_columns] = (
    model_df[lag_columns]
    .fillna(0)
)

#### Save Final Time Series Dataset

In [89]:
print("Shape:", model_df.shape)

model_df.info()

Shape: (29240, 31)
<class 'pandas.DataFrame'>
RangeIndex: 29240 entries, 0 to 29239
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   hospital_id           29240 non-null  str           
 1   ward                  29240 non-null  str           
 2   bed_type              29240 non-null  str           
 3   date                  29240 non-null  datetime64[us]
 4   total_beds            29240 non-null  int64         
 5   staffed_beds          29240 non-null  int64         
 6   occupied_beds         29240 non-null  float64       
 7   closed_beds           29240 non-null  float64       
 8   occupancy_rate        29240 non-null  float64       
 9   available_beds        29240 non-null  float64       
 10  daily_admissions      29240 non-null  float64       
 11  avg_los_hours         29240 non-null  float64       
 12  emergency_admissions  29240 non-null  float64       
 13  planned_

In [90]:
missing = (
    model_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing)

hospital_id             0
daily_ed_arrivals       0
day_of_week             0
rolling_7_day_avg       0
lag_7_occupancy         0
lag_1_occupancy         0
bed_pressure            0
occupancy_rate_model    0
over_capacity           0
total_bed_capacity      0
specialty_emphasis      0
tier                    0
city                    0
hospital_name           0
scheduled_surgeries     0
staffing_ratio          0
ward                    0
actual_staff            0
planned_staff           0
emergency_admissions    0
avg_los_hours           0
daily_admissions        0
available_beds          0
occupancy_rate          0
closed_beds             0
occupied_beds           0
staffed_beds            0
total_beds              0
date                    0
bed_type                0
occupancy_change        0
dtype: int64


In [91]:
duplicates = model_df.duplicated(
    subset=[
        "hospital_id",
        "ward",
        "bed_type",
        "date"
    ]
).sum()

print("Duplicate records:", duplicates)

Duplicate records: 0


In [92]:
print(
    model_df["occupancy_rate"].describe()
)

count    29240.000000
mean         0.742548
std          0.308865
min          0.000000
25%          0.564394
50%          0.833333
75%          0.992188
max          1.159722
Name: occupancy_rate, dtype: float64


In [93]:
over_capacity = model_df[
    model_df["occupancy_rate"] > 1
]

print(over_capacity.shape)

(3953, 31)


#### Feature Engineering

##### Calendar Features

In [94]:
model_df["date"] = pd.to_datetime(model_df["date"])

model_df["year"] = model_df["date"].dt.year

model_df["month"] = model_df["date"].dt.month

model_df["day"] = model_df["date"].dt.day

model_df["day_of_week"] = model_df["date"].dt.dayofweek

model_df["week_of_year"] = model_df["date"].dt.isocalendar().week.astype(int)

model_df["quarter"] = model_df["date"].dt.quarter

model_df["is_weekend"] = (
    model_df["day_of_week"] >= 5
).astype(int)

##### Bed Pressure

In [95]:
model_df["bed_pressure"] = (
    model_df["occupied_beds"]
    -
    model_df["staffed_beds"]
)

##### Available Bed Percentage

In [96]:
model_df["available_bed_pct"] = (
    model_df["available_beds"]
    /
    model_df["staffed_beds"]
)

##### Staff Shortage

In [97]:
model_df["staff_shortage"] = (
    model_df["planned_staff"]
    -
    model_df["actual_staff"]
)

##### Emergency Admission Ratio

In [98]:
model_df["emergency_ratio"] = (
    model_df["emergency_admissions"]
    /
    model_df["daily_admissions"]
)

In [99]:
model_df["emergency_ratio"] = (
    model_df["emergency_ratio"]
    .fillna(0)
)

##### Surgery Pressure

In [100]:
model_df["surgery_per_staff"] = (
    model_df["scheduled_surgeries"]
    /
    model_df["actual_staff"]
)

model_df["surgery_per_staff"] = (
    model_df["surgery_per_staff"]
    .fillna(0)
)

In [101]:
model_df.head()

,hospital_id,ward,bed_type,date,total_beds,staffed_beds,occupied_beds,closed_beds,occupancy_rate,available_beds,daily_admissions,avg_los_hours,emergency_admissions,planned_staff,actual_staff,staffing_ratio,daily_ed_arrivals,scheduled_surgeries,hospital_name,city,tier,specialty_emphasis,total_bed_capacity,over_capacity,occupancy_rate_model,bed_pressure,lag_1_occupancy,lag_7_occupancy,rolling_7_day_avg,day_of_week,occupancy_change,year,month,day,week_of_year,quarter,is_weekend,available_bed_pct,staff_shortage,emergency_ratio,surgery_per_staff
0,HHN-BIR-01,Cardiology Ward,Standard,2024-01-01,42,42,3.083333,0.0,0.073413,38.916667,6.0,83.650000,6.0,20,20,1.00,80,0.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.073413,-38.916667,0.000000,0.0,0.0,0,0.000000,2024,1,1,1,1,0,0.926587,0,1.000000,0.000000
1,HHN-BIR-01,Cardiology Ward,Standard,2024-01-02,42,42,10.333333,0.0,0.246032,31.666667,10.0,101.810000,9.0,20,17,0.85,83,3.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.246032,-31.666667,0.073413,0.0,0.0,1,0.172619,2024,1,2,1,1,0,0.753968,3,0.900000,0.176471
2,HHN-BIR-01,Cardiology Ward,Standard,2024-01-03,42,42,17.750000,0.0,0.422619,24.250000,7.0,69.628571,6.0,20,20,1.00,72,2.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.422619,-24.250000,0.246032,0.0,0.0,2,0.176587,2024,1,3,1,1,0,0.577381,0,0.857143,0.100000
3,HHN-BIR-01,Cardiology Ward,Standard,2024-01-04,42,42,18.958333,0.0,0.451389,23.041667,7.0,66.457143,5.0,20,20,1.00,95,9.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.451389,-23.041667,0.422619,0.0,0.0,3,0.028770,2024,1,4,1,1,0,0.548611,0,0.714286,0.450000
4,HHN-BIR-01,Cardiology Ward,Standard,2024-01-05,42,42,21.500000,0.0,0.511905,20.500000,6.0,93.983333,2.0,20,18,0.90,80,14.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0,0.511905,-20.500000,0.451389,0.0,0.0,4,0.060516,2024,1,5,1,1,0,0.488095,2,0.333333,0.777778


##### Save time series dataset

In [ ]:
model_df.reset_index(inplace=True)

model_df.to_csv(
    "../data/processed/bed_occupancy_timeseries.csv",
    index=False
)